In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:95% !important;}
div.cell.code_cell.rendered{width:95%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:22pt;}
.inner_cell{font-size:22pt;}
div.text_cell_render pre code {font-size:22pt; line-height:30px;}
div.output {font-size:20pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:22pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render li{font-size:20pt;padding:5px; line-height:30px;}
table.dataframe{font-size:22px;}
</style>
"""))

**<font size="6" color="red">ch5. LSTM(Long Short-Term Memory ; RNN)으로 영화평 구분하기</font>**
- imdb의 5만개 영화 감상평(독립변수) - 부정/긍정(종속변수)

# 1. 패키지 import

In [2]:
import numpy as np
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from time import time # 70.1.1부터 현재까지 몇초가 지났는지

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, GRU, Dense#, Bidirectional
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.metrics import confusion_matrix, recall_score, precision_score, f1_score
import pandas as pd

# 2. 하이퍼 파라미터 설정(이 파라미터를 바꾸면 모델 score나 학습속도에 차이)

In [3]:
MY_WORDS = 10000 # imdb 데이터안의 단어 수
MY_LENGTH = 80 # 영화평 단어수 80개까지만 독립변수 (200추천)
MY_EMBED = 32  # 임베딩 layer의 출력 차원(256추천)

MY_EPOCH = 10 # 반복 학습 수(fit 20추천)
MY_BATCH = 200 # 매번 가져오는 데이터수 
# 불용어 인덱스 설정(빈도수 상위 40개는 제외 : the, a, is등)
SKIP_TOP = 40

# 3. 데이터 불러오기

In [4]:
(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=MY_WORDS) # MY_WORDS(10000)개

17464789/17464789 [==============================] - 1s 0us/step


In [11]:
print('학습용 데이터 shape :', X_train.shape, y_train.shape)
print('학습용 입력 데이터 샘플 :', X_train[0], '-', type(X_train[0]), '-', len(X_train[0]))
print('학습용 타겟 데이터 샘플(0:부정/1:긍정) :', y_train[0])

print('시험용 데이터 shape :', X_test.shape, y_test.shape)
print('시험용 입력 데이터 샘플 :', X_test[0], '-', type(X_test[0]), '-', len(X_test[0]))
print('시험용 타겟 데이터 샘플(0:부정/1:긍정) :', y_test[0])

학습용 데이터 shape : (25000,) (25000,)
학습용 입력 데이터 샘플 : [1, 14, 22, 16, 43, 530, 973, 1622, 1385, 65, 458, 4468, 66, 3941, 4, 173, 36, 256, 5, 25, 100, 43, 838, 112, 50, 670, 2, 9, 35, 480, 284, 5, 150, 4, 172, 112, 167, 2, 336, 385, 39, 4, 172, 4536, 1111, 17, 546, 38, 13, 447, 4, 192, 50, 16, 6, 147, 2025, 19, 14, 22, 4, 1920, 4613, 469, 4, 22, 71, 87, 12, 16, 43, 530, 38, 76, 15, 13, 1247, 4, 22, 17, 515, 17, 12, 16, 626, 18, 2, 5, 62, 386, 12, 8, 316, 8, 106, 5, 4, 2223, 5244, 16, 480, 66, 3785, 33, 4, 130, 12, 16, 38, 619, 5, 25, 124, 51, 36, 135, 48, 25, 1415, 33, 6, 22, 12, 215, 28, 77, 52, 5, 14, 407, 16, 82, 2, 8, 4, 107, 117, 5952, 15, 256, 4, 2, 7, 3766, 5, 723, 36, 71, 43, 530, 476, 26, 400, 317, 46, 7, 4, 2, 1029, 13, 104, 88, 4, 381, 15, 297, 98, 32, 2071, 56, 26, 141, 6, 194, 7486, 18, 4, 226, 22, 21, 134, 476, 26, 480, 5, 144, 30, 5535, 18, 51, 36, 28, 224, 92, 25, 104, 4, 226, 65, 16, 38, 1334, 88, 12, 16, 283, 5, 16, 4472, 113, 103, 32, 15, 16, 5345, 19, 178, 32] - <class '

In [18]:
# X_train의 길이 및 평균길이
print([len(x) for x in X_train[:10]])
np.array([len(x) for x in X_train]).mean()

[218, 189, 141, 550, 147, 43, 123, 562, 233, 130]


238.71364

In [20]:
# 부/긍정 갯수 확인
print('학습용 데이터의 긍정 갯수 :', y_train.sum())
print('테스트용 데이터의 긍정 갯수 :', y_test.sum())

학습용 데이터의 긍정 갯수 : 12500
테스트용 데이터의 긍정 갯수 : 12500


In [24]:
# 부/긍정 갯수 확인
pd.Series(y_train).value_counts()#.sort_index()

1    12500
0    12500
dtype: int64

# 4. 문자 단어 -> 정수

In [32]:
word_to_id = imdb.get_word_index() # dict (단어:id값) : 빈도수가 높은 단어를 id앞에
print(word_to_id['movie'])
print(word_to_id['film'])
print(word_to_id['the'])
# 정수 : 단어 dict
id_to_word = {}
for word, id in word_to_id.items():
    id_to_word[id] = word
print(id_to_word[17])
print(id_to_word[19])
print(id_to_word[1])

17
19
1
movie
film
the


In [34]:
# 감성분성에서 필요없는 조사, 관사
print([id_to_word.get(i) for i in range(1, 40)])

['the', 'and', 'a', 'of', 'to', 'is', 'br', 'in', 'it', 'i', 'this', 'that', 'was', 'as', 'for', 'with', 'movie', 'but', 'film', 'on', 'not', 'you', 'are', 'his', 'have', 'he', 'be', 'one', 'all', 'at', 'by', 'an', 'they', 'who', 'so', 'from', 'like', 'her', 'or']


In [52]:
msg = 'What a wonderful movie'
msg = msg.lower().split()
# 1: 리뷰시작할때 무조건 추가, 2:1000개문자가 짤려서 잘못 읽어옴, 3:padding처리
data = [1] + [word_to_id.get(m, -1)+3 for m in msg]
print('원 후기 내용 :', msg)
print('encoding된 data :', data)
data_msg = ' '.join([id_to_word.get(d-3, '??') for d in data])
print('data로 추정된 msg :', data_msg)

원 후기 내용 : ['what', 'a', 'wonderful', 'movie']
encoding된 data : [1, 51, 6, 389, 20]
data로 추정된 msg : ?? what a wonderful movie


# 5. 숫자 영화평 -> 자연어 영화평 함수